In [1]:
# Cell 1: Cài đặt thư viện cần thiết cho PIPNet, TorchLM, OpenCV & HuggingFace trên Google Colab
import importlib.util
import os
import platform
import subprocess
import sys

if platform.system() == "Linux":
    print("Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...")
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq libgl1 libglib2.0-0 >/dev/null 2>&1",
        shell=True,
    )

_needed_pkgs = [
    pkg for mod, pkg in [
        ("torchlm", "torchlm"),
        ("huggingface_hub", "huggingface_hub"),
        ("cv2", "opencv-python"),
        ("tqdm", "tqdm"),
        ("yaml", "pyyaml"),
        ("scipy", "scipy"),
    ]
    if importlib.util.find_spec(mod) is None
]
if _needed_pkgs:
    print(f"Đang cài đặt các thư viện còn thiếu: {_needed_pkgs}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", *_needed_pkgs
    ])
else:
    print("Tất cả các thư viện cần thiết đã được cài đặt sẵn.")

import torch

_ram_gb = 0.0
if hasattr(os, "sysconf") and "SC_PAGE_SIZE" in os.sysconf_names and "SC_PHYS_PAGES" in os.sysconf_names:
    _ram_gb = (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) / (1024 ** 3)
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()} | RAM: {_ram_gb:.1f} GB")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Cài đặt hoàn tất! Môi trường đã sẵn sàng.")


Đang cấu hình gói hệ thống trên Linux (Ubuntu/Colab)...
Tất cả các thư viện cần thiết đã được cài đặt sẵn.
PyTorch: 2.14.1+cu130 | CUDA available: True | RAM: 53.0 GB
GPU: NVIDIA L4
Cài đặt hoàn tất! Môi trường đã sẵn sàng.


In [ ]:
# Cell 2: Tải / Định vị Dataset data_processed_v1 (KSS 1-9 từ HuggingFace hoặc Local)
import csv
from collections import Counter
from pathlib import Path

HF_DATASET_REPO = "nyvantran/data_processed_v1"

candidates = [
    Path("/content/data_processed_v1"),
    Path("./data_processed_v1"),
    Path(r"D:\Project\DATN\driver-guardian\ai\data_processed_v1"),
    Path(r"E:\LSTM\data_processed_v1"),
]

for snap_root in (
    Path("/root/.cache/huggingface/hub/datasets--nyvantran--data_processed_v1/snapshots"),
    Path.home() / ".cache/huggingface/hub/datasets--nyvantran--data_processed_v1/snapshots",
):
    if snap_root.is_dir():
        snaps = sorted(
            (p for p in snap_root.iterdir() if p.is_dir()),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        candidates = snaps + candidates

detected_dataset_dir = None
for c in candidates:
    if (c / "dataset_merged_split.csv").is_file() or (c / "dataset_merged_split.json").is_file():
        detected_dataset_dir = c
        print(f"Đã tìm thấy Dataset data_processed_v1 tại: {detected_dataset_dir}")
        break

if detected_dataset_dir is None:
    print(f"Không tìm thấy manifest cục bộ. Đang tải {HF_DATASET_REPO} từ HuggingFace Hub...")
    from huggingface_hub import snapshot_download

    path_origin = Path(snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset"))
    for cand in (path_origin, path_origin / "data_processed_v1"):
        if (cand / "dataset_merged_split.csv").is_file() or (cand / "dataset_merged_split.json").is_file():
            detected_dataset_dir = cand
            break
    if detected_dataset_dir is None:
        detected_dataset_dir = path_origin
    print(f"Tải thành công! Đường dẫn dữ liệu: {detected_dataset_dir}")

_csv_manifest = detected_dataset_dir / "dataset_merged_split.csv"
if _csv_manifest.is_file():
    with open(_csv_manifest, "r", encoding="utf-8-sig", newline="") as f:
        _rows = list(csv.DictReader(f))
    print(f"Manifest: {len(_rows)} mẫu | Splits: {dict(Counter(r.get('split', '').strip().lower() for r in _rows))}")


In [2]:
# Cell 3: Import các thư viện cần thiết & Cấu hình môi trường PyTorch CUDA
from __future__ import annotations
import csv
import gc
import json
import logging
import math
import os
import random
import shutil
import sys
import tempfile
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor, Future
from dataclasses import dataclass, field, asdict, fields
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

# Monkey patch cho scipy >= 1.14 để tương thích với torchlm metrics
import scipy.integrate
if not hasattr(scipy.integrate, "simps"):
    scipy.integrate.simps = scipy.integrate.simpson

# Đảm bảo mã hóa UTF-8 trên console
if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Giới hạn số luồng nội bộ của OpenCV để tránh tranh chấp khóa luồng (thread contention)
# khi giải mã nhiều video song song bằng ThreadPoolExecutor
cv2.setNumThreads(1)

# Tắt cuDNN benchmark vì chunk cuối của mỗi video có số lượng mặt (cur_B) thay đổi liên tục;
# tránh việc cuDNN phải chạy lại thuật toán autotune mất vài giây ở mỗi kích thước batch lẻ mới.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    if hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "allow_tf32"):
        torch.backends.cudnn.allow_tf32 = True

# Cấu hình môi trường nạp DLL PyTorch CUDA trên Windows
torch_lib_path = os.path.join(os.path.dirname(sys.executable), "Library", "bin")
if os.path.exists(torch_lib_path):
    os.environ["PATH"] = torch_lib_path + os.pathsep + os.environ.get("PATH", "")
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

logger = logging.getLogger("DriverGuardian.LiteFat_v1.Extract")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(name)s: %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


In [3]:
# Cell 4: Hệ thống cấu hình trung tâm (ExtractConfig - đồng bộ 10 FPS với LiteFat_v1)
def _detect_system_ram_gb() -> float:
    if hasattr(os, "sysconf") and "SC_PAGE_SIZE" in os.sysconf_names and "SC_PHYS_PAGES" in os.sysconf_names:
        return (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) / (1024 ** 3)
    return 16.0


def _default_cache_dir() -> str:
    if Path("/content").exists():
        return "/content/LiteFat_v1/cache_pt"
    if Path(r"E:\LSTM").exists():
        return r"E:\LSTM\LiteFat_v1\cache_pt"
    return "./LiteFat_v1/cache_pt"


_SYSTEM_RAM_GB = _detect_system_ram_gb()


@dataclass
class ExtractConfig:
    """
    Cấu hình trích xuất cache .pt cho LiteFat_v1 (KSS 1-9 @ 10 FPS):
      - Đồng bộ time_interval = 0.1s (10 FPS) và window_sec = None với TrainConfig.
      - Tự động điều chỉnh prefetch_videos & num_io_workers theo RAM khả dụng để tránh OOM
        với các video dài 130s-315s của data_processed_v1.
    """
    # 1. Cấu hình thư mục dữ liệu & Manifest
    dataset_dir: str = r"/root/.cache/huggingface/hub/datasets--nyvantran--data_processed_v1/snapshots/7b90773741e01f4d19ac6618d724690b5abff1ba"
    manifest_filename: str = "dataset_merged_split.csv"

    # 2. Cấu hình lấy mẫu thời gian (Temporal Sampling - khớp 100% với train_1.ipynb)
    # time_interval = 0.1: lấy mẫu ở 10 FPS
    # window_sec = None: trích xuất toàn bộ video để model chạy sliding window 60s
    time_interval: float = 0.1
    window_sec: Optional[float] = None

    # 3. Cấu hình hình học khuôn mặt & PIPNet
    letterbox_size: Tuple[int, int] = (640, 640)
    face_size: Tuple[int, int] = (224, 224)
    pipnet_backbone: str = "resnet18"
    conf_threshold: float = 0.5
    nms_threshold: float = 0.3
    bbox_margin: float = 0.15
    box_smooth_alpha: float = 0.7

    # 4. Cấu hình tăng tốc phần cứng (GPU Batching, FP16 & RAM-Safe Video Prefetching)
    use_fp16: bool = True
    face_batch_size: int = 64
    lmk_batch_size: int = 128
    num_io_workers: int = 2 if _SYSTEM_RAM_GB >= 24.0 else 0
    prefetch_videos: int = 2 if _SYSTEM_RAM_GB >= 24.0 else 1

    # 5. Cấu hình lưu trữ Cache (.pt - khớp với TrainConfig.cache_dir)
    cache_dir: str = field(default_factory=_default_cache_dir)
    cache_splits: List[str] = field(default_factory=lambda: ["train", "val", "test"])
    cache_save_every: int = 25        # Lưu tạm atomic save sau mỗi 25 video (tập train v1 có 518 video dài)
    resume_cache: bool = True
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    def __post_init__(self):
        if isinstance(self.letterbox_size, (list, tuple)) and len(self.letterbox_size) == 2:
            self.letterbox_size = (int(self.letterbox_size[0]), int(self.letterbox_size[1]))
        if isinstance(self.face_size, (list, tuple)) and len(self.face_size) == 2:
            self.face_size = (int(self.face_size[0]), int(self.face_size[1]))
        self.time_interval = float(self.time_interval)
        self.face_batch_size = max(1, int(self.face_batch_size))
        self.lmk_batch_size = max(1, int(self.lmk_batch_size))
        self.num_io_workers = max(0, int(self.num_io_workers))
        self.prefetch_videos = max(1, int(self.prefetch_videos))
        self.cache_save_every = max(1, int(self.cache_save_every))

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def save(self, path: Union[str, Path]) -> None:
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        with open(p, "w", encoding="utf-8") as f:
            json.dump(self.to_dict(), f, indent=4, ensure_ascii=False)

    @classmethod
    def load(cls, path: Union[str, Path]) -> "ExtractConfig":
        p = Path(path)
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        valid_keys = {fld.name for fld in fields(cls)}
        filtered = {k: v for k, v in data.items() if k in valid_keys}
        return cls(**filtered)


In [4]:
# Cell 5: Toàn bộ Định nghĩa Mô-đun Tiền Xử Lý, PIPNet Extractor & LiteFatDataset Độc Lập
# Đảm bảo chạy 100% độc lập trên Google Colab / Kaggle mà không cần tải thêm file ultis.py hay dataset.py
import csv
import os
import random
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

if hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset

# ---------------------------------------------------------------------
# 1. TIỆN ÍCH LETTERBOX & KHỞI TẠO PIPNET
# ---------------------------------------------------------------------
def _parse_size(size: Optional[Union[int, Tuple[int, int], List[int]]]) -> Tuple[int, int]:
    """Chuẩn hóa kích thước về tuple (width, height)."""
    if size is None:
        return (640, 640)
    if isinstance(size, int):
        return (size, size)
    if isinstance(size, (tuple, list)) and len(size) == 2:
        return (int(size[0]), int(size[1]))
    raise ValueError(f"Kích thước không hợp lệ: {size}. Phải là int hoặc tuple (w, h).")


def letterbox(
        image: np.ndarray,
        new_size: Optional[Union[int, Tuple[int, int], List[int]]] = (640, 640),
        color: Tuple[int, int, int] = (114, 114, 114),
) -> np.ndarray:
    """Resize ảnh giữ nguyên tỷ lệ khung hình gốc với đệm viền về kích thước đích (w, h)."""
    target_w, target_h = _parse_size(new_size)
    if target_w <= 0 or target_h <= 0:
        return image

    h, w = image.shape[:2]
    if w == target_w and h == target_h:
        return image

    scale = min(target_w / float(w), target_h / float(h))
    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR

    if nw == target_w and nh == target_h:
        return cv2.resize(image, (nw, nh), interpolation=interp)

    canvas = np.full((target_h, target_w, 3), color, dtype=image.dtype)
    x, y = (target_w - nw) // 2, (target_h - nh) // 2
    cv2.resize(image, (nw, nh), dst=canvas[y:y + nh, x:x + nw], interpolation=interp)
    return canvas


def init_pipnet_68_gpu(backbone: str = "resnet18"):
    """Khởi tạo pipeline FaceBoxesV2 + PIPNet (68 điểm chuẩn 300W) trên CUDA / CPU."""
    import torchlm
    from torchlm.tools import faceboxesv2
    from torchlm.models import pipnet

    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Đang khởi tạo PIPNet ({backbone} - 68 điểm 300W) trên thiết bị: {device.upper()}")
    torchlm.runtime.bind(faceboxesv2(device=device))
    torchlm.runtime.bind(
        pipnet(
            backbone=backbone,
            pretrained=True,
            num_nb=10,
            num_lms=68,
            net_stride=32,
            input_size=256,
            meanface_type="300w",
            map_location=device
        )
    )
    logger.info("Đã nạp xong trọng số mô hình PIPNet 68!")


def compute_iou(box_a: np.ndarray, box_b: np.ndarray) -> float:
    """Tính Intersection over Union (IoU) giữa 2 hộp giới hạn [x1, y1, x2, y2]."""
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return float(inter / union)


# ---------------------------------------------------------------------
# 2. BỘ TRÍCH XUẤT PIPNET TỐI ƯU HÓA GPU (FAST PIPNET EXTRACTOR)
# ---------------------------------------------------------------------
class FastPIPNetExtractor:
    """
    Bộ trích xuất 68 điểm mốc khuôn mặt và tọa độ điểm cắt khung mặt cho LiteFat:
      1. Batched FaceBoxesV2 trên GPU: Hỗ trợ FP16, truyền uint8 qua PCIe, và batched_nms (1 lần sync/chunk).
      2. Temporal Driver Tracking: Theo dõi và khóa khuôn mặt tài xế ổn định theo IoU.
      3. Batched PIPNet 300W trên GPU: Giải mã NRM vector hóa qua bảng ánh xạ với FP16.
      4. Landmark-Guided BBox Refinement: Kết hợp cực trị mốc và bbox vector hóa, nội suy tuyến tính & EMA.
      5. Xuất tọa độ điểm cắt crop_boxes [T, 4] dạng nguyên [x1, y1, x2, y2] trong letterbox.
    """
    def __init__(
            self,
            backbone: str = "resnet18",
            device: Optional[str] = None,
            letterbox_size: Tuple[int, int] = (640, 640),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            nms_threshold: float = 0.3,
            bbox_margin: float = 0.15,
            box_smooth_alpha: float = 0.7,
            use_fp16: bool = True,
            face_batch_size: int = 64,
            lmk_batch_size: int = 128,
    ):
        self.device = torch.device(
            device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = float(conf_threshold)
        self.nms_threshold = float(nms_threshold)
        self.bbox_margin = float(bbox_margin)
        self.box_smooth_alpha = float(box_smooth_alpha)
        self.use_fp16 = bool(use_fp16) and (self.device.type == "cuda")
        self.face_batch_size = max(1, int(face_batch_size))
        self.lmk_batch_size = max(1, int(lmk_batch_size))

        from torchlm.runtime._wrappers import RuntimeWrapper
        if getattr(RuntimeWrapper, "face_base", None) is None or getattr(RuntimeWrapper, "landmarks_base", None) is None:
            init_pipnet_68_gpu(backbone=backbone)

        self.face_base = RuntimeWrapper.face_base
        self.face_net = self.face_base.net.to(self.device)
        self.face_net.eval()

        self.lmk_base = RuntimeWrapper.landmarks_base
        self.lmk_net = self.lmk_base.to(self.device)
        self.lmk_net.eval()

        target_w, target_h = self.letterbox_size
        from torchlm.tools._faceboxesv2 import PriorBox
        pbox = PriorBox(self.face_base.cfg, image_size=(target_h, target_w))
        self.cached_priors = pbox.forward().to(self.device)
        self.cached_variances = self.face_base.cfg["variance"]
        self.cached_scale = torch.tensor(
            [target_w, target_h, target_w, target_h], dtype=torch.float32, device=self.device
        )
        self.faceboxes_mean = torch.tensor(
            [104.0, 117.0, 123.0], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        self.pipnet_mean = torch.tensor(
            [0.485, 0.456, 0.406], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)
        self.pipnet_std = torch.tensor(
            [0.229, 0.224, 0.225], dtype=torch.float32, device=self.device
        ).view(1, 3, 1, 1)

        rev1 = self.lmk_base.reverse_index1
        rev2 = self.lmk_base.reverse_index2
        num_nb = self.lmk_base.num_nb
        self.rev_indices = torch.tensor(
            [r1 * num_nb + r2 for r1, r2 in zip(rev1, rev2)],
            dtype=torch.long,
            device=self.device,
        )

    @torch.no_grad()
    def _detect_faces_batched(
            self,
            frames: Union[np.ndarray, List[np.ndarray]],
            max_chunk: Optional[int] = None,
    ) -> List[np.ndarray]:
        T = len(frames)
        if T == 0:
            return []
        chunk_size = max_chunk if max_chunk is not None else self.face_batch_size
        all_dets: List[np.ndarray] = [np.empty((0, 5), dtype=np.float32) for _ in range(T)]
        priors_xy = self.cached_priors[None, :, :2]
        priors_wh = self.cached_priors[None, :, 2:]
        v0, v1 = self.cached_variances[0], self.cached_variances[1]
        is_4d_arr = isinstance(frames, np.ndarray) and frames.ndim == 4

        for s in range(0, T, chunk_size):
            if is_4d_arr:
                frames_np = np.ascontiguousarray(frames[s:s + chunk_size])
            else:
                frames_np = np.stack(frames[s:s + chunk_size], axis=0)

            # Truyền uint8 liên tục qua PCIe (nhẹ hơn 4x so với float32), sau đó permute trên GPU
            tensor_u8 = torch.from_numpy(frames_np).to(self.device, non_blocking=True)
            tensor = tensor_u8.permute(0, 3, 1, 2).to(dtype=torch.float32).sub_(self.faceboxes_mean)

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=self.use_fp16):
                loc, conf = self.face_net(tensor)

            loc = loc.float()
            scores = conf[:, :, 1].float()

            boxes_xy = priors_xy + loc[:, :, :2] * v0 * priors_wh
            boxes_wh = priors_wh * torch.exp(loc[:, :, 2:] * v1)
            b_x1y1 = boxes_xy - boxes_wh / 2.0
            b_x2y2 = b_x1y1 + boxes_wh
            decoded_boxes = torch.cat([b_x1y1, b_x2y2], dim=-1) * self.cached_scale

            # Lọc ngưỡng và chạy batched_nms cho toàn bộ chunk để chỉ đồng bộ GPU->CPU 1 lần duy nhất
            mask = scores >= self.conf_threshold
            batch_idx, anchor_idx = torch.where(mask)
            if batch_idx.numel() == 0:
                continue

            bx_valid = decoded_boxes[batch_idx, anchor_idx]
            sc_valid = scores[batch_idx, anchor_idx]
            keep = torchvision.ops.batched_nms(bx_valid, sc_valid, batch_idx, iou_threshold=self.nms_threshold)

            kept_batch = batch_idx[keep].cpu().numpy()
            kept_dets = torch.cat([bx_valid[keep], sc_valid[keep, None]], dim=-1).cpu().numpy()

            # Gom nhóm theo chỉ số frame bằng stable sort (giữ nguyên thứ tự điểm tin cậy giảm dần)
            order = np.argsort(kept_batch, kind="stable")
            sorted_batch = kept_batch[order]
            unique_b, start_indices, counts = np.unique(sorted_batch, return_index=True, return_counts=True)
            for b_val, st_idx, cnt in zip(unique_b, start_indices, counts):
                sel = order[st_idx:st_idx + min(50, int(cnt))]
                all_dets[s + int(b_val)] = kept_dets[sel]

        return all_dets

    def _select_driver_faces(
            self,
            all_dets: List[np.ndarray],
            target_w: int,
            target_h: int
    ) -> Tuple[List[Optional[np.ndarray]], List[bool]]:
        T = len(all_dets)
        chosen_bboxes: List[Optional[np.ndarray]] = [None] * T
        found: List[bool] = [False] * T
        prev_box: Optional[np.ndarray] = None
        prev_box_area = 1.0

        for t in range(T):
            dets = all_dets[t]
            if len(dets) == 0:
                continue
            best_idx = 0
            if prev_box is None:
                best_score = -1.0
                for i, d in enumerate(dets):
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = d[4] * (area ** 0.5)
                    if metric > best_score:
                        best_score = metric
                        best_idx = i
            else:
                best_match = -1.0
                for i, d in enumerate(dets):
                    iou = compute_iou(d[:4], prev_box[:4])
                    area = max(0.0, d[2] - d[0]) * max(0.0, d[3] - d[1])
                    metric = (iou * 2.0) + (d[4] * 0.5) + (min(1.0, area / (prev_box_area + 1e-6)) * 0.3)
                    if metric > best_match:
                        best_match = metric
                        best_idx = i

            chosen = dets[best_idx].copy()
            chosen_bboxes[t] = chosen
            found[t] = True
            prev_box = chosen
            prev_box_area = max(1.0, (prev_box[2] - prev_box[0]) * (prev_box[3] - prev_box[1]))
        return chosen_bboxes, found

    @torch.no_grad()
    def _extract_landmarks_batched(
            self,
            frames: Union[np.ndarray, List[np.ndarray]],
            chosen_bboxes: List[Optional[np.ndarray]],
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        T = len(frames)
        target_w, target_h = self.letterbox_size
        extend = 0.2
        valid_indices = [t for t in range(T) if found[t] and chosen_bboxes[t] is not None]

        landmarks_norm = np.ones((T, 68, 3), dtype=np.float32)
        landmarks_px = np.zeros((T, 68, 2), dtype=np.float32)
        if not valid_indices:
            return landmarks_norm, landmarks_px

        n_valid = len(valid_indices)
        crop_coords = np.empty((n_valid, 4), dtype=np.float32)
        crops_bgr = np.empty((n_valid, 256, 256, 3), dtype=np.uint8)

        for i, idx in enumerate(valid_indices):
            bbox = chosen_bboxes[idx]
            bx1, by1, bx2, by2 = bbox[:4]
            bw = bx2 - bx1 + 1.0
            bh = by2 - by1 + 1.0
            cx1 = max(0, min(target_w - 2, int(bx1 - bw * extend / 2.0)))
            cy1 = max(0, min(target_h - 2, int(by1 + bh * extend / 2.0)))
            cx2 = max(cx1 + 1, min(target_w, int(bx2 + bw * extend / 2.0)))
            cy2 = max(cy1 + 1, min(target_h, int(by2 + bh * extend / 2.0)))
            crop_coords[i] = (cx1, cy1, float(cx2 - cx1), float(cy2 - cy1))

            # Resize trực tiếp trên mảng BGR liên tục vào vùng đệm đích (không lật kênh trên CPU)
            cv2.resize(frames[idx][cy1:cy2, cx1:cx2], (256, 256), dst=crops_bgr[i], interpolation=cv2.INTER_LINEAR)

        max_crop_chunk = self.lmk_batch_size
        batched_lms_list = []
        norm_factor = 1.0 * self.lmk_base.input_size / self.lmk_base.net_stride
        num_nb = self.lmk_base.num_nb
        max_len = self.lmk_base.max_len

        for s in range(0, n_valid, max_crop_chunk):
            chunk_u8 = torch.from_numpy(crops_bgr[s:s + max_crop_chunk]).to(self.device, non_blocking=True)
            # Permute [B, H, W, C] -> [B, C, H, W], lật kênh BGR -> RGB bằng flip(1) và chuẩn hóa trên GPU
            crops_tensor = chunk_u8.permute(0, 3, 1, 2).flip(1).to(dtype=torch.float32).div_(255.0)
            crops_tensor.sub_(self.pipnet_mean).div_(self.pipnet_std)

            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=self.use_fp16):
                out_cls, out_x, out_y, out_nb_x, out_nb_y = self.lmk_net(crops_tensor)

            out_cls = out_cls.float()
            out_x = out_x.float()
            out_y = out_y.float()
            out_nb_x = out_nb_x.float()
            out_nb_y = out_nb_y.float()

            cur_B, num_lms, H_map, W_map = out_cls.shape
            out_cls = out_cls.view(cur_B, num_lms, -1)
            max_ids = torch.argmax(out_cls, dim=2)
            max_ids_col = max_ids.unsqueeze(-1)

            out_x_flat = out_x.view(cur_B, num_lms, -1)
            out_y_flat = out_y.view(cur_B, num_lms, -1)
            x_sel = torch.gather(out_x_flat, 2, max_ids_col).squeeze(2)
            y_sel = torch.gather(out_y_flat, 2, max_ids_col).squeeze(2)

            max_ids_nb = max_ids_col.repeat(1, 1, num_nb).view(cur_B, num_lms * num_nb, 1)
            out_nb_x_flat = out_nb_x.view(cur_B, num_lms * num_nb, -1)
            out_nb_y_flat = out_nb_y.view(cur_B, num_lms * num_nb, -1)
            nb_x_sel = torch.gather(out_nb_x_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)
            nb_y_sel = torch.gather(out_nb_y_flat, 2, max_ids_nb).view(cur_B, num_lms, num_nb)

            pred_x = ((max_ids % W_map).float() + x_sel) / norm_factor
            pred_y = (torch.floor(max_ids.float() / W_map) + y_sel) / norm_factor
            pred_nb_x = ((max_ids % W_map).float().unsqueeze(-1) + nb_x_sel) / norm_factor
            pred_nb_y = (torch.floor(max_ids.float() / W_map).unsqueeze(-1) + nb_y_sel) / norm_factor
            pred_nb_x_flat = pred_nb_x.view(cur_B, -1)
            pred_nb_y_flat = pred_nb_y.view(cur_B, -1)

            nb_x_rev = pred_nb_x_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)
            nb_y_rev = pred_nb_y_flat[:, self.rev_indices].view(cur_B, num_lms, max_len)

            final_x = torch.mean(torch.cat([pred_x.unsqueeze(-1), nb_x_rev], dim=-1), dim=-1)
            final_y = torch.mean(torch.cat([pred_y.unsqueeze(-1), nb_y_rev], dim=-1), dim=-1)
            chunk_lms = torch.stack([final_x, final_y], dim=-1).cpu().numpy()
            batched_lms_list.append(chunk_lms)

        batched_lms = np.concatenate(batched_lms_list, axis=0) if batched_lms_list else np.empty((0, 68, 2), dtype=np.float32)

        # Ánh xạ ngược tọa độ từ crop về khung hình letterbox (vector hóa toàn bộ)
        valid_arr = np.asarray(valid_indices, dtype=np.intp)
        cx1 = crop_coords[:, 0:1]
        cy1 = crop_coords[:, 1:2]
        cw = crop_coords[:, 2:3]
        ch = crop_coords[:, 3:4]
        abs_x = batched_lms[:, :, 0] * cw + cx1
        abs_y = batched_lms[:, :, 1] * ch + cy1
        scores_valid = np.array([float(chosen_bboxes[idx][4]) for idx in valid_indices], dtype=np.float32)[:, None]

        landmarks_px[valid_arr, :, 0] = abs_x
        landmarks_px[valid_arr, :, 1] = abs_y
        landmarks_norm[valid_arr, :, 0] = abs_x / float(target_w)
        landmarks_norm[valid_arr, :, 1] = abs_y / float(target_h)
        landmarks_norm[valid_arr, :, 2] = scores_valid

        return landmarks_norm, landmarks_px

    def compute_face_crop_boxes(
            self,
            chosen_bboxes: List[Optional[np.ndarray]],
            landmarks_px: np.ndarray,
            found: List[bool],
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Tính toán tọa độ điểm cắt khuôn mặt vuông dạng nguyên [T, 4] [x1, y1, x2, y2]."""
        T = len(chosen_bboxes)
        target_w, target_h = self.letterbox_size
        raw_boxes = np.zeros((T, 4), dtype=np.float32)

        valid_indices = [t for t in range(T) if found[t] and chosen_bboxes[t] is not None]
        if valid_indices:
            valid_arr = np.asarray(valid_indices, dtype=np.intp)
            bboxes_v = np.stack([chosen_bboxes[t][:4] for t in valid_indices], axis=0)
            lms_v = landmarks_px[valid_arr]
            lx1 = np.min(lms_v[:, :, 0], axis=1)
            ly1 = np.min(lms_v[:, :, 1], axis=1)
            lx2 = np.max(lms_v[:, :, 0], axis=1)
            ly2 = np.max(lms_v[:, :, 1], axis=1)

            fx1 = np.minimum(bboxes_v[:, 0], lx1)
            fy1 = np.minimum(bboxes_v[:, 1], ly1)
            fx2 = np.maximum(bboxes_v[:, 2], lx2)
            fy2 = np.maximum(bboxes_v[:, 3], ly2)
            fw = fx2 - fx1
            fh = fy2 - fy1
            cx1 = fx1 - fw * self.bbox_margin
            cy1 = fy1 - fh * self.bbox_margin
            cx2 = fx2 + fw * self.bbox_margin
            cy2 = fy2 + fh * self.bbox_margin
            center_x = (cx1 + cx2) * 0.5
            center_y = (cy1 + cy2) * 0.5
            half_side = np.maximum(cx2 - cx1, cy2 - cy1) * 0.5

            raw_boxes[valid_arr, 0] = center_x - half_side
            raw_boxes[valid_arr, 1] = center_y - half_side
            raw_boxes[valid_arr, 2] = center_x + half_side
            raw_boxes[valid_arr, 3] = center_y + half_side

            first_idx = valid_indices[0]
            if first_idx > 0:
                raw_boxes[:first_idx] = raw_boxes[first_idx]
            last_idx = valid_indices[-1]
            if last_idx < T - 1:
                raw_boxes[last_idx + 1:] = raw_boxes[last_idx]
            for i in range(len(valid_indices) - 1):
                idx_a = valid_indices[i]
                idx_b = valid_indices[i + 1]
                if idx_b - idx_a > 1:
                    box_a = raw_boxes[idx_a]
                    box_b = raw_boxes[idx_b]
                    steps = np.arange(1, idx_b - idx_a, dtype=np.float32)[:, None] / float(idx_b - idx_a)
                    raw_boxes[idx_a + 1:idx_b] = (1.0 - steps) * box_a + steps * box_b
        else:
            side = min(target_w, target_h) * 0.8
            cx, cy = target_w / 2.0, target_h / 2.0
            raw_boxes[:] = [cx - side / 2.0, cy - side / 2.0, cx + side / 2.0, cy + side / 2.0]

        smoothed_boxes = np.zeros_like(raw_boxes)
        smoothed_boxes[0] = raw_boxes[0]
        alpha = self.box_smooth_alpha
        one_minus_alpha = 1.0 - alpha
        for t in range(1, T):
            smoothed_boxes[t] = alpha * raw_boxes[t] + one_minus_alpha * smoothed_boxes[t - 1]

        rounded = np.rint(smoothed_boxes).astype(np.int32)
        x1 = np.clip(rounded[:, 0], 0, target_w - 2)
        y1 = np.clip(rounded[:, 1], 0, target_h - 2)
        x2 = np.maximum(x1 + 1, np.clip(rounded[:, 2], 0, target_w))
        y2 = np.maximum(y1 + 1, np.clip(rounded[:, 3], 0, target_h))
        crop_boxes = np.stack([x1, y1, x2, y2], axis=-1)

        return crop_boxes, smoothed_boxes

    def _crop_faces_for_litefat(
            self,
            frames: Union[np.ndarray, List[np.ndarray]],
            crop_boxes: np.ndarray,
    ) -> torch.Tensor:
        T = len(frames)
        crop_w, crop_h = self.face_size
        face_crops = np.zeros((T, crop_h, crop_w, 3), dtype=np.uint8)

        for t in range(T):
            x1, y1, x2, y2 = crop_boxes[t]
            crop = frames[t][y1:y2, x1:x2, :]  # BGR
            if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
                crop = frames[t]
            resized = cv2.resize(crop, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
            face_crops[t] = resized[:, :, ::-1]  # Chuyển sang RGB

        face_tensor = torch.from_numpy(face_crops).permute(0, 3, 1, 2).float() / 255.0
        norm_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        norm_std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        face_tensor = (face_tensor - norm_mean) / norm_std
        return face_tensor

    def extract_from_frames(
            self,
            frames: Union[np.ndarray, List[np.ndarray]],
            crop_faces: bool = True,
            return_crop_boxes: bool = False,
    ) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool]],
        Tuple[Optional[torch.Tensor], torch.Tensor, List[Optional[np.ndarray]], List[bool], np.ndarray],
    ]:
        target_w, target_h = self.letterbox_size
        if isinstance(frames, np.ndarray) and frames.ndim == 4 and frames.shape[1] == target_h and frames.shape[2] == target_w:
            uniform_frames: Union[np.ndarray, List[np.ndarray]] = frames
        else:
            uniform_list = []
            for f in frames:
                if f.shape[1] != target_w or f.shape[0] != target_h:
                    uniform_list.append(letterbox(f, new_size=(target_w, target_h)))
                else:
                    uniform_list.append(f)
            uniform_frames = np.stack(uniform_list, axis=0) if uniform_list else np.empty((0, target_h, target_w, 3), dtype=np.uint8)

        all_dets = self._detect_faces_batched(uniform_frames)
        chosen_bboxes, found = self._select_driver_faces(all_dets, target_w, target_h)
        landmarks_norm, landmarks_px = self._extract_landmarks_batched(uniform_frames, chosen_bboxes, found)
        crop_boxes, _ = self.compute_face_crop_boxes(chosen_bboxes, landmarks_px, found)

        face_frames_tensor = None
        if crop_faces:
            face_frames_tensor = self._crop_faces_for_litefat(uniform_frames, crop_boxes)

        landmarks_tensor = torch.from_numpy(landmarks_norm).float()
        if return_crop_boxes:
            return face_frames_tensor, landmarks_tensor, chosen_bboxes, found, crop_boxes
        return face_frames_tensor, landmarks_tensor, chosen_bboxes, found


# ---------------------------------------------------------------------
# 3. ĐỌC NHANH KHUNG HÌNH TỪ VIDEO (ZERO-COPY 4D CONTIGUOUS BUFFER)
# ---------------------------------------------------------------------
def fast_read_video_frames(
        video_path: Union[str, Path],
        target_indices: np.ndarray,
        letterbox_size: Tuple[int, int] = (640, 640),
) -> Tuple[np.ndarray, float, int]:
    """
    Giải mã các khung hình mục tiêu từ video và resize trực tiếp vào vùng đệm 4D liên tục
    shape (T, target_h, target_w, 3) kiểu uint8. Loại bỏ hoàn toàn việc cấp phát mảng tạm mỗi frame.
    """
    if len(target_indices) == 0:
        target_indices = np.array([0], dtype=int)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Không thể mở tệp video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    target_w, target_h = _parse_size(letterbox_size)
    num_targets = len(target_indices)

    frames_out = np.full((num_targets, target_h, target_w, 3), 114, dtype=np.uint8)
    curr_frame = int(target_indices[0])
    if curr_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)

    geom_ready = False
    direct_copy = False
    nw = nh = x_off = y_off = 0
    interp = cv2.INTER_LINEAR
    read_count = 0

    try:
        for i, raw_idx in enumerate(target_indices):
            target_idx = int(raw_idx)
            # Nếu chỉ số frame trùng với frame vừa đọc trước đó, copy trực tiếp trong vùng đệm
            if i > 0 and target_idx < curr_frame:
                frames_out[i] = frames_out[i - 1]
                read_count += 1
                continue

            while curr_frame < target_idx:
                if not cap.grab():
                    break
                curr_frame += 1

            ret, frame = cap.read()
            if not ret or frame is None:
                break
            curr_frame += 1

            if not geom_ready:
                h, w = frame.shape[:2]
                if w == target_w and h == target_h:
                    direct_copy = True
                else:
                    scale = min(target_w / float(w), target_h / float(h))
                    nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
                    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
                    x_off, y_off = (target_w - nw) // 2, (target_h - nh) // 2
                geom_ready = True

            if direct_copy:
                frames_out[i] = frame
            else:
                cv2.resize(
                    frame,
                    (nw, nh),
                    dst=frames_out[i, y_off:y_off + nh, x_off:x_off + nw],
                    interpolation=interp,
                )
            read_count += 1
    finally:
        cap.release()

    if read_count == 0:
        raise IOError(f"Không đọc được khung hình nào từ: {video_path}")
    if read_count < num_targets:
        frames_out[read_count:] = frames_out[read_count - 1]

    return frames_out, fps, total_frames


# ---------------------------------------------------------------------
# 4. LỚP LITEFAT DATASET (HỖ TRỢ TRÍCH XUẤT .PT & PREFETCH ĐA LUỒNG)
# ---------------------------------------------------------------------
def _safe_int(val: Any, default: int = 0) -> int:
    if val is None or str(val).strip() == "":
        return default
    return int(round(float(val)))


def _safe_float(val: Any, default: float = 0.0) -> float:
    if val is None or str(val).strip() == "":
        return default
    return float(val)


class LiteFatDataset(Dataset):
    """
    PyTorch Dataset trích xuất cache .pt cho LiteFat_v1 (KSS 1-9 @ 10 FPS):
      - Đọc manifest (.csv hoặc .json) đồng bộ hoàn toàn với dataset.py của train_1.ipynb.
      - Tách biệt bước đọc video trên CPU (read_video_only) và bước suy luận GPU (extract_from_read_video)
        để hỗ trợ giải mã video đa luồng bất đồng bộ (Multi-thread Prefetching).
      - Đóng gói đầy đủ metadata KSS (key, video_id, kss_score, kss_raw, sample_weight, target_indices...).
    """
    def __init__(
            self,
            dataset_dir: Union[str, Path] = "/content/data_processed_v1",
            split: str = "train",
            time_interval: float = 0.1,
            window_sec: Optional[float] = None,
            manifest_filename: str = "dataset_merged_split.csv",
            letterbox_size: Tuple[int, int] = (640, 640),
            face_size: Tuple[int, int] = (224, 224),
            conf_threshold: float = 0.5,
            bbox_margin: float = 0.15,
            return_meta: bool = False,
            return_raw_frames: bool = False,
            use_cache: bool = False,
            cache_dir: Optional[Union[str, Path]] = None,
            extractor: Optional[FastPIPNetExtractor] = None,
            device: Optional[str] = None,
            extract_crops_only: bool = False,
    ):
        split = split.lower().strip()
        if split not in ("train", "val", "test"):
            raise ValueError(f"Split không hợp lệ: '{split}'. Chỉ chấp nhận 'train', 'val', hoặc 'test'.")

        self.root = Path(dataset_dir)
        self.split = split
        self.time_interval = float(time_interval) if time_interval is not None else 0.1
        self.window_sec = float(window_sec) if window_sec is not None else None
        self.manifest_filename = manifest_filename
        self.letterbox_size = _parse_size(letterbox_size)
        self.face_size = _parse_size(face_size)
        self.conf_threshold = conf_threshold
        self.bbox_margin = bbox_margin
        self.return_meta = bool(return_meta)
        self.return_raw_frames = bool(return_raw_frames)
        self.use_cache = bool(use_cache)
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None
        self.extract_crops_only = bool(extract_crops_only)

        self.samples = self._load_samples()
        if len(self.samples) == 0:
            logger.warning(f"Không tìm thấy mẫu video nào trong {self.root} cho split='{self.split}'")

        self.cached_data: Optional[Dict[str, Any]] = None
        if self.use_cache and self.cache_dir is not None:
            self._load_cache()

        if extractor is not None:
            self.extractor = extractor
        else:
            self.extractor = FastPIPNetExtractor(
                backbone="resnet18",
                device=device,
                letterbox_size=self.letterbox_size,
                face_size=self.face_size,
                conf_threshold=self.conf_threshold,
                bbox_margin=self.bbox_margin,
            )

    def _resolve_manifest_path(self) -> Path:
        candidate = Path(self.manifest_filename)
        if not candidate.is_absolute():
            candidate = self.root / self.manifest_filename
        if candidate.is_file() and candidate.suffix.lower() in (".csv", ".json"):
            return candidate
        alt_suffix = ".json" if candidate.suffix.lower() == ".csv" else ".csv"
        alt_candidate = candidate.with_suffix(alt_suffix)
        if alt_candidate.is_file():
            return alt_candidate
        if self.root.is_dir():
            for ext in ("*.csv", "*.json"):
                matches = sorted(self.root.glob(ext))
                if matches:
                    return matches[0]
        raise FileNotFoundError(
            f"Không tìm thấy tệp manifest (.csv hoặc .json) '{self.manifest_filename}' trong thư mục: {self.root}"
        )

    def _parse_manifest_record(self, row: Dict[str, Any], default_split: Optional[str] = None) -> Optional[Dict[str, Any]]:
        raw_split = row.get("split")
        split_val = str(raw_split if raw_split is not None else (default_split or "")).strip().lower()
        if split_val != self.split:
            return None

        rel_path = row.get("rel_path") or row.get("processed_path") or row.get("path")
        if not rel_path or str(rel_path).strip() == "":
            return None

        norm_rel = Path(str(rel_path).strip().replace("\\", "/")).as_posix()
        rel_obj = Path(norm_rel)
        full_path = rel_obj if rel_obj.is_absolute() else (self.root / norm_rel)

        raw_lbl = row.get("label")
        if raw_lbl is None or str(raw_lbl).strip() == "":
            raw_lbl = row.get("kss_score", row.get("kss_raw", 1))
        label_int = _safe_int(raw_lbl, default=1)
        kss_score = _safe_int(row.get("kss_score"), default=label_int)
        kss_raw = _safe_float(row.get("kss_raw"), default=float(kss_score))
        kss_class = max(0, min(8, label_int - 1))
        sample_weight = _safe_float(row.get("sample_weight"), default=1.0)

        return {
            "key": norm_rel,
            "path": full_path,
            "orig_path": str(row.get("orig_path", "") or ""),
            "label": label_int,
            "kss_score": kss_score,
            "kss_raw": kss_raw,
            "kss_class": kss_class,
            "sample_weight": sample_weight,
            "video_id": str(row.get("video_id") or rel_obj.stem),
            "subject_id": str(row.get("subject_id", "") or "unknown"),
            "session_id": str(row.get("session_id", "") or ""),
            "condition": str(row.get("condition", "") or ""),
            "dataset": str(row.get("dataset", "") or "unknown"),
            "fps": _safe_float(row.get("fps"), default=30.0),
            "frame_count": _safe_int(row.get("frame_count"), default=0),
            "width": _safe_int(row.get("width"), default=0),
            "height": _safe_int(row.get("height"), default=0),
            "duration_sec": _safe_float(row.get("duration_sec"), default=0.0),
        }

    def _load_samples(self) -> List[Dict[str, Any]]:
        manifest_path = self._resolve_manifest_path()
        ext = manifest_path.suffix.lower()
        samples: List[Dict[str, Any]] = []
        if ext == ".csv":
            with open(manifest_path, mode="r", encoding="utf-8-sig", newline="") as f:
                for row in csv.DictReader(f):
                    parsed = self._parse_manifest_record(row)
                    if parsed is not None:
                        samples.append(parsed)
        elif ext == ".json":
            with open(manifest_path, mode="r", encoding="utf-8-sig") as f:
                data = json.load(f)
            default_split: Optional[str] = None
            if isinstance(data, dict) and isinstance(data.get("splits"), dict):
                records = data["splits"].get(self.split, [])
                default_split = self.split
            elif isinstance(data, dict) and ("samples" in data or "data" in data):
                records = data.get("samples") or data.get("data") or []
            elif isinstance(data, list):
                records = data
            else:
                raise ValueError(f"Cấu trúc tệp JSON manifest không hợp lệ: {manifest_path}")
            for row in records:
                if isinstance(row, dict):
                    parsed = self._parse_manifest_record(row, default_split=default_split)
                    if parsed is not None:
                        samples.append(parsed)
        return samples

    def _load_cache(self) -> None:
        cache_path = self.cache_dir / f"{self.split}.pt"
        if not cache_path.is_file():
            logger.warning(f"[Cache] Không tìm thấy tệp cache {cache_path}. Sẽ fallback trích xuất on-the-fly.")
            return
        try:
            payload = torch.load(cache_path, map_location="cpu", weights_only=False)
            if isinstance(payload, dict) and "samples" in payload:
                self.cached_data = payload["samples"]
            elif isinstance(payload, dict):
                self.cached_data = payload
            logger.info(f"[Cache] Đã nạp thành công cache {cache_path.name} với {len(self.cached_data)} mẫu.")
        except Exception as e:
            logger.warning(f"[Cache] Lỗi khi nạp cache {cache_path}: {e}. Bỏ qua cache.")

    def _build_timestamps(self, start_sec: float, span_sec: float) -> np.ndarray:
        """Tạo mảng mốc thời gian khớp 100% với _build_timestamps của train_1.ipynb."""
        if span_sec <= 0.0:
            return np.array([start_sec], dtype=np.float64)
        end_bound = max(self.time_interval * 0.5, span_sec) - 1e-6
        timestamps = start_sec + np.arange(0.0, max(1e-7, end_bound), self.time_interval, dtype=np.float64)
        if len(timestamps) == 0:
            timestamps = np.array([start_sec], dtype=np.float64)
        return timestamps

    def _get_frame_indices(self, total_frames: int, fps: float) -> np.ndarray:
        if total_frames <= 0:
            return np.array([0], dtype=int)
        fps = fps if fps > 0 else 30.0
        duration_sec = total_frames / fps

        if self.time_interval == 0:
            if self.window_sec is None:
                return np.arange(total_frames, dtype=int)
            win_sec = min(self.window_sec, duration_sec)
            num_consec = max(1, min(int(round(win_sec * fps)), total_frames))
            max_start = max(0, total_frames - num_consec)
            start_idx = random.randint(0, max_start) if (self.split == "train" and max_start > 0) else max_start // 2
            return np.arange(start_idx, start_idx + num_consec, dtype=int)

        if self.window_sec is None:
            timestamps = self._build_timestamps(0.0, duration_sec)
        else:
            win = min(self.window_sec, duration_sec)
            max_start = max(0.0, duration_sec - win)
            start_sec = random.uniform(0.0, max_start) if (self.split == "train" and max_start > 0) else max_start / 2.0
            timestamps = self._build_timestamps(start_sec, win)
        indices = np.rint(timestamps * fps).astype(int)
        return np.clip(indices, 0, total_frames - 1)

    def __len__(self) -> int:
        return len(self.samples)

    def read_video_only(self, index: int) -> Tuple[np.ndarray, float, int, np.ndarray]:
        """
        Bước 1 (CPU-only, Thread-safe): Mở video, tính target_indices và giải mã vào vùng đệm 4D letterbox.
        Có thể gọi song song từ nhiều luồng trong ThreadPoolExecutor.
        """
        sample = self.samples[index]
        fps = sample["fps"]
        total_frames = sample["frame_count"]
        if fps <= 0 or total_frames <= 0:
            cap = cv2.VideoCapture(str(sample["path"]))
            fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 300)
            cap.release()
            sample["fps"] = fps
            sample["frame_count"] = total_frames

        target_indices = self._get_frame_indices(total_frames, fps)
        raw_frames, actual_fps, read_total = fast_read_video_frames(
            sample["path"], target_indices, letterbox_size=self.letterbox_size
        )
        if total_frames <= 0 and read_total > 0:
            total_frames = read_total
        return raw_frames, actual_fps, total_frames, target_indices

    def extract_from_read_video(
            self,
            index: int,
            raw_frames: np.ndarray,
            actual_fps: float,
            total_frames: int,
            target_indices: np.ndarray,
    ):
        """
        Bước 2 (GPU Inference): Nhận vùng đệm raw_frames từ bước 1 và trích xuất 68 điểm mốc + crop_boxes.
        """
        sample = self.samples[index]
        key = sample["key"]
        crop_faces = not self.extract_crops_only
        face_frames, landmarks, bboxes, found, crop_boxes = self.extractor.extract_from_frames(
            raw_frames,
            crop_faces=crop_faces,
            return_crop_boxes=True,
        )
        label = torch.tensor(sample["label"], dtype=torch.long)

        T_frames = len(found)
        bboxes_tensor = torch.zeros((T_frames, 5), dtype=torch.float32)
        for t_idx, b in enumerate(bboxes):
            if b is not None and len(b) >= 4:
                b_arr = np.asarray(b, dtype=np.float32)
                bboxes_tensor[t_idx, :min(5, len(b_arr))] = torch.from_numpy(b_arr[:min(5, len(b_arr))])
        found_tensor = torch.tensor(found, dtype=torch.bool)
        crop_boxes_tensor = torch.from_numpy(crop_boxes).int()

        meta = {
            "key": key,
            "video_id": sample["video_id"],
            "subject_id": str(sample.get("subject_id", "unknown")),
            "session_id": str(sample.get("session_id", "")),
            "condition": str(sample.get("condition", "")),
            "dataset": str(sample.get("dataset", "unknown")),
            "path": str(sample["path"]),
            "orig_path": str(sample.get("orig_path", "")),
            "split": self.split,
            "label": int(sample["label"]),
            "kss_score": int(sample.get("kss_score", sample["label"])),
            "kss_raw": float(sample.get("kss_raw", sample["label"])),
            "kss_class": int(sample.get("kss_class", max(0, min(8, int(sample["label"]) - 1)))),
            "sample_weight": float(sample.get("sample_weight", 1.0)),
            "fps": float(actual_fps),
            "width": int(sample.get("width", 0)),
            "height": int(sample.get("height", 0)),
            "total_frames": int(total_frames),
            "duration_sec": float(sample["duration_sec"]) if sample.get("duration_sec", 0) > 0 else (float(total_frames / actual_fps) if actual_fps > 0 else 0.0),
            "target_indices": torch.from_numpy(target_indices).long(),
            "num_frames": int(landmarks.shape[0]),
            "found_faces_ratio": float(sum(found)) / float(max(1, len(found))),
            "letterbox_size": self.letterbox_size,
            "crop_boxes": crop_boxes_tensor,
            "bboxes": bboxes_tensor,
            "found": found_tensor,
            "raw_bboxes_list": bboxes,
        }

        if self.return_raw_frames and self.return_meta:
            return face_frames, landmarks, label, meta, list(raw_frames)
        if self.return_meta:
            return face_frames, landmarks, label, meta
        return face_frames, landmarks, label

    def __getitem__(self, index: int) -> Union[
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]],
        Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any], List[np.ndarray]],
    ]:
        sample = self.samples[index]
        key = sample["key"]

        # 1. Đường nhanh: Nạp từ RAM Cache nếu có
        if self.cached_data is not None and key in self.cached_data:
            item = self.cached_data[key]
            face_frames = item.get("face_frames", None)
            landmarks = item["landmarks"]
            label = torch.tensor(sample["label"], dtype=torch.long)
            meta = {
                "key": key,
                "video_id": sample["video_id"],
                "subject_id": str(sample.get("subject_id") or (item.get("meta", {}).get("subject_id", "unknown"))),
                "dataset": str(sample.get("dataset") or (item.get("meta", {}).get("dataset", "unknown"))),
                "path": str(sample["path"]),
                "split": self.split,
                "label": int(sample["label"]),
                "fps": float(sample["fps"]),
                "num_frames": landmarks.shape[0],
                "cached": True,
            }
            if "crop_boxes" in item:
                meta["crop_boxes"] = item["crop_boxes"]
            if "bboxes" in item:
                meta["bboxes"] = item["bboxes"]
            if "found" in item:
                meta["found"] = item["found"]
            if "target_indices" in item:
                meta["target_indices"] = item["target_indices"]
            if "meta" in item and isinstance(item["meta"], dict):
                meta.update(item["meta"])

            if self.return_meta:
                return face_frames, landmarks, label, meta
            return face_frames, landmarks, label

        # 2. Đường chuẩn: Đọc video và trích xuất on-the-fly
        raw_frames, actual_fps, total_frames, target_indices = self.read_video_only(index)
        return self.extract_from_read_video(index, raw_frames, actual_fps, total_frames, target_indices)

print("Đã nạp toàn bộ các thành phần LiteFatDataset, FastPIPNetExtractor (FP16 + Batched NMS), Zero-Copy Video Reader thành công!")


Đã nạp toàn bộ các thành phần LiteFatDataset, FastPIPNetExtractor (FP16 + Batched NMS), Zero-Copy Video Reader thành công!


In [5]:
# Cell 6: Lớp quản lý trích xuất cache hướng đối tượng (LiteFatPtCacheBuilder)
class LiteFatPtCacheBuilder:
    """
    Bộ điều phối trích xuất và lưu cache .pt cho LiteFat:
      1. Khởi tạo LiteFatDataset với extract_crops_only=True để tối ưu tốc độ và bộ nhớ RAM.
      2. Sử dụng Bounded Multi-Thread Prefetching (ThreadPoolExecutor) để giải mã video trên CPU
         gối đầu song song trong khi GPU liên tục chạy suy luận FaceBoxesV2 + PIPNet.
      3. Trích xuất 68 điểm mốc, tọa độ điểm cắt khuôn mặt crop_boxes [T, 4], bboxes [T, 5], found [T].
      4. Hỗ trợ Auto-Resume (bỏ qua video đã trích xuất), Atomic Save an toàn chống hỏng file, và kiểm tra tính toàn vẹn (verify).
    """
    @staticmethod
    def seed_everything(seed: int = 42) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    @staticmethod
    def configure_logging(log_file: Union[Path, str]) -> None:
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        logger.setLevel(logging.INFO)
        logger.propagate = False
        for handler in logger.handlers[:]:
            handler.close()
            logger.removeHandler(handler)

        formatter = logging.Formatter("[%(asctime)s][%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
        file_handler = logging.FileHandler(log_file, encoding="utf-8")
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

        stream_handler = logging.StreamHandler(sys.stdout)
        stream_handler.setFormatter(formatter)
        logger.addHandler(stream_handler)

    @staticmethod
    def atomic_save(state: Any, path: Union[str, Path], sync_disk: bool = False) -> None:
        """Lưu tệp an toàn (Atomic Save qua file tạm) với pickle_protocol=4 để tối ưu tốc độ ghi."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fd, tmp_file = tempfile.mkstemp(dir=path.parent, prefix=f".{path.name}.", suffix=".tmp")
        try:
            with os.fdopen(fd, "wb") as f:
                torch.save(state, f, pickle_protocol=4)
                f.flush()
                if sync_disk:
                    os.fsync(f.fileno())
            os.replace(tmp_file, path)
        finally:
            Path(tmp_file).unlink(missing_ok=True)

    @staticmethod
    def verify_cache_file(path: Union[str, Path]) -> Dict[str, Any]:
        """Kiểm tra tính toàn vẹn của một file cache .pt (cấu trúc, shape landmarks [T, 68, 3], crop_boxes [T, 4], bboxes [T, 5])."""
        path = Path(path)
        report: Dict[str, Any] = {"path": str(path), "ok": False, "num_samples": 0, "errors": []}
        if not path.is_file():
            report["errors"].append("Không tìm thấy file")
            return report
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except Exception as exc:
            report["errors"].append(f"Không đọc được file: {exc}")
            return report

        samples = payload.get("samples", {})
        report["num_samples"] = len(samples)
        total_frames = 0
        report["time_interval"] = payload.get("time_interval", payload.get("meta", {}).get("time_interval"))
        for key, item in samples.items():
            lm = item.get("landmarks")
            cb = item.get("crop_boxes")
            bx = item.get("bboxes")
            fd = item.get("found")
            meta = item.get("meta", {})
            t_idx = meta.get("target_indices")
            if lm is None or lm.ndim != 3 or lm.shape[0] <= 0 or lm.shape[1] != 68 or lm.shape[2] != 3:
                report["errors"].append(f"{key}: shape landmarks không hợp lệ (kỳ vọng [T>0, 68, 3])")
            elif cb is None or cb.ndim != 2 or cb.shape != (lm.shape[0], 4):
                report["errors"].append(f"{key}: shape crop_boxes không khớp [T, 4]")
            elif bx is None or bx.ndim != 2 or bx.shape != (lm.shape[0], 5):
                report["errors"].append(f"{key}: shape bboxes không khớp [T, 5]")
            elif fd is None or fd.ndim != 1 or fd.shape[0] != lm.shape[0]:
                report["errors"].append(f"{key}: shape found không khớp [T]")
            elif t_idx is None or len(t_idx) != lm.shape[0]:
                report["errors"].append(f"{key}: target_indices không khớp chiều dài T")
            elif not torch.isfinite(lm).all():
                report["errors"].append(f"{key}: landmarks có giá trị NaN/Inf")
            elif "video_id" not in meta or "path" not in meta:
                report["errors"].append(f"{key}: thiếu metadata video định danh")
            else:
                total_frames += lm.shape[0]

        report["total_frames"] = total_frames
        report["ok"] = len(report["errors"]) == 0
        return report

    def __init__(self, cfg: Optional[ExtractConfig] = None):
        self.cfg = cfg if cfg is not None else ExtractConfig()
        self.cfg.__post_init__()

        self.cache_dir = Path(self.cfg.cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.configure_logging(self.cache_dir / "extract.log")
        self.seed_everything(self.cfg.seed)
        self.cfg.save(self.cache_dir / "config.json")

        # Khởi tạo 1 bộ trích xuất PIPNet dùng chung cho mọi split để tiết kiệm GPU VRAM
        self.extractor = FastPIPNetExtractor(
            backbone=self.cfg.pipnet_backbone,
            device=self.cfg.device,
            letterbox_size=self.cfg.letterbox_size,
            face_size=self.cfg.face_size,
            conf_threshold=self.cfg.conf_threshold,
            nms_threshold=self.cfg.nms_threshold,
            bbox_margin=self.cfg.bbox_margin,
            box_smooth_alpha=self.cfg.box_smooth_alpha,
            use_fp16=self.cfg.use_fp16,
            face_batch_size=self.cfg.face_batch_size,
            lmk_batch_size=self.cfg.lmk_batch_size,
        )

        logger.info("=" * 70)
        logger.info("KHỞI TẠO PIPELINE TRÍCH XUẤT .PT CHO LITEFAT DATASET (TURBO GPU + ASYNC I/O)")
        logger.info("=" * 70)
        logger.info(f"Dataset root              : {self.cfg.dataset_dir}")
        logger.info(f"Thư mục cache (.pt)       : {self.cache_dir}")
        logger.info(f"Các split                 : {self.cfg.cache_splits}")
        logger.info(f"Time interval             : {self.cfg.time_interval}s")
        logger.info(f"Window sec                : {self.cfg.window_sec} (None = toàn bộ video)")
        logger.info(f"Letterbox size (w, h)     : {self.cfg.letterbox_size}")
        logger.info(f"Face size (crop)          : {self.cfg.face_size}")
        logger.info(f"Thiết bị trích xuất       : {self.cfg.device} (FP16={self.extractor.use_fp16})")
        logger.info(f"GPU Batch (Face / Lmk)    : {self.cfg.face_batch_size} / {self.cfg.lmk_batch_size}")
        logger.info(f"CPU Video I/O Workers     : {self.cfg.num_io_workers} (Prefetch={self.cfg.prefetch_videos} videos)")

    def _new_cache(self, split: str) -> Dict[str, Any]:
        meta_header = {
            "format_version": 1,
            "split": split,
            "time_interval": float(self.cfg.time_interval),
            "window_sec": self.cfg.window_sec,
            "letterbox_size": tuple(self.cfg.letterbox_size),
            "face_size": tuple(self.cfg.face_size),
        }
        return {
            **meta_header,
            "bbox_margin": float(self.cfg.bbox_margin),
            "box_smooth_alpha": float(self.cfg.box_smooth_alpha),
            "num_samples": 0,
            "samples": {},
            "meta": meta_header,
        }

    def _load_existing_cache(self, split: str, path: Path) -> Dict[str, Any]:
        """Nạp file .pt đã có để Auto-Resume; khởi tạo mới nếu chưa có hoặc cấu hình không khớp."""
        if not (self.cfg.resume_cache and path.is_file()):
            return self._new_cache(split)
        try:
            payload = torch.load(path, map_location="cpu", weights_only=False)
        except Exception as exc:
            logger.warning(f"Không đọc được {path.name} ({exc}). Trích xuất lại từ đầu.")
            return self._new_cache(split)

        if abs(float(payload.get("time_interval", -1.0)) - float(self.cfg.time_interval)) > 1e-9:
            logger.warning(f"{path.name} có time_interval khác cấu hình hiện tại. Trích xuất lại từ đầu.")
            return self._new_cache(split)
        cached_lbs = payload.get("letterbox_size", ())
        if tuple(cached_lbs) != tuple(self.cfg.letterbox_size):
            logger.warning(
                f"{path.name} tạo với letterbox_size={cached_lbs} khác cấu hình hiện tại {self.cfg.letterbox_size}. Trích xuất lại từ đầu."
            )
            return self._new_cache(split)
        return payload

    def _pack_sample_entry(
            self,
            key: str,
            split: str,
            sample_meta: Dict[str, Any],
            landmarks: torch.Tensor,
            label: torch.Tensor,
            meta: Dict[str, Any],
    ) -> Dict[str, Any]:
        return {
            "landmarks": landmarks.cpu(),                         # Tensor [T, 68, 3]
            "crop_boxes": meta["crop_boxes"].cpu(),               # Tensor [T, 4] (x1, y1, x2, y2)
            "bboxes": meta["bboxes"].cpu(),                       # Tensor [T, 5] (x1, y1, x2, y2, conf)
            "found": meta["found"].cpu(),                         # Tensor [T] (bool)
            "label": int(label.item()),
            "meta": {
                "key": key,
                "video_id": str(meta["video_id"]),
                "subject_id": str(meta.get("subject_id", sample_meta.get("subject_id", "unknown"))),
                "session_id": str(meta.get("session_id", sample_meta.get("session_id", ""))),
                "condition": str(meta.get("condition", sample_meta.get("condition", ""))),
                "dataset": str(meta.get("dataset", sample_meta.get("dataset", "unknown"))),
                "path": str(meta["path"]),
                "orig_path": str(meta.get("orig_path", sample_meta.get("orig_path", ""))),
                "split": split,
                "label": int(label.item()),
                "kss_score": int(meta.get("kss_score", label.item())),
                "kss_raw": float(meta.get("kss_raw", label.item())),
                "kss_class": int(meta.get("kss_class", max(0, min(8, int(label.item()) - 1)))),
                "sample_weight": float(meta.get("sample_weight", sample_meta.get("sample_weight", 1.0))),
                "fps": float(meta["fps"]),
                "width": int(meta.get("width", sample_meta.get("width", 0))),
                "height": int(meta.get("height", sample_meta.get("height", 0))),
                "total_frames": int(meta["total_frames"]),
                "duration_sec": float(meta["duration_sec"]),
                "target_indices": meta["target_indices"].cpu(),   # Tensor [T]
                "num_frames": int(meta["num_frames"]),
                "found_faces_ratio": float(meta["found_faces_ratio"]),
                "letterbox_size": tuple(meta["letterbox_size"]),
            }
        }

    def extract_split(self, split: str) -> Dict[str, Any]:
        """Trích xuất toàn bộ video của một split với cơ chế đọc video đa luồng bất đồng bộ và lưu {cache_dir}/{split}.pt."""
        path = self.cache_dir / f"{split}.pt"
        ds = LiteFatDataset(
            dataset_dir=self.cfg.dataset_dir,
            split=split,
            time_interval=self.cfg.time_interval,
            window_sec=self.cfg.window_sec,
            manifest_filename=self.cfg.manifest_filename,
            letterbox_size=self.cfg.letterbox_size,
            face_size=self.cfg.face_size,
            conf_threshold=self.cfg.conf_threshold,
            bbox_margin=self.cfg.bbox_margin,
            return_meta=True,
            extract_crops_only=True,
            use_cache=False,
            extractor=self.extractor,
            device=self.cfg.device,
        )

        stats: Dict[str, Any] = {
            "split": split, "total": len(ds), "cached_before": 0, "new": 0, "failed": [],
            "frames": 0, "seconds": 0.0, "interrupted": False, "path": str(path),
        }
        if len(ds) == 0:
            logger.warning(f"Không có mẫu nào trong split '{split}'. Bỏ qua.")
            return stats

        cache = self._load_existing_cache(split, path)
        done = cache["samples"]
        todo_indices = [i for i, s in enumerate(ds.samples) if s["key"] not in done]
        stats["cached_before"] = len(ds) - len(todo_indices)

        logger.info("=" * 60)
        logger.info(f"BẮT ĐẦU TRÍCH XUẤT SPLIT [{split.upper()}]: tổng {len(ds)} | đã có {stats['cached_before']} | cần trích xuất {len(todo_indices)}")
        logger.info("=" * 60)

        if todo_indices:
            t0 = time.time()
            since_save = 0
            use_async_io = self.cfg.num_io_workers > 0 and len(todo_indices) > 1

            try:
                if use_async_io:
                    max_prefetch = max(1, self.cfg.prefetch_videos)
                    todo_iter = iter(todo_indices)
                    pending: deque[Tuple[int, Future]] = deque()

                    with ThreadPoolExecutor(max_workers=self.cfg.num_io_workers) as executor:
                        for _ in range(min(max_prefetch, len(todo_indices))):
                            nxt = next(todo_iter, None)
                            if nxt is not None:
                                pending.append((nxt, executor.submit(ds.read_video_only, nxt)))

                        pbar = tqdm(total=len(todo_indices), desc=f"Extracting [{split}]", unit="video", dynamic_ncols=True)
                        try:
                            while pending:
                                idx, fut = pending.popleft()
                                nxt = next(todo_iter, None)
                                if nxt is not None:
                                    pending.append((nxt, executor.submit(ds.read_video_only, nxt)))

                                sample_meta = ds.samples[idx]
                                key = sample_meta["key"]
                                raw_frames = None
                                try:
                                    raw_frames, actual_fps, total_frames, target_indices = fut.result()
                                    _, landmarks, label, meta = ds.extract_from_read_video(
                                        idx, raw_frames, actual_fps, total_frames, target_indices
                                    )
                                except Exception as exc:
                                    logger.error(f"Lỗi trích xuất video '{sample_meta['path']}': {exc}")
                                    stats["failed"].append({"key": key, "error": str(exc)})
                                    pbar.update(1)
                                    continue
                                finally:
                                    del raw_frames

                                done[key] = self._pack_sample_entry(key, split, sample_meta, landmarks, label, meta)
                                stats["new"] += 1
                                stats["frames"] += int(landmarks.shape[0])
                                since_save += 1
                                pbar.update(1)

                                if since_save >= self.cfg.cache_save_every:
                                    cache["num_samples"] = len(done)
                                    self.atomic_save(cache, path, sync_disk=False)
                                    gc.collect()
                                    since_save = 0
                        finally:
                            for _, f_rem in pending:
                                f_rem.cancel()
                            pbar.close()
                else:
                    for idx in tqdm(todo_indices, desc=f"Extracting [{split}]", unit="video", dynamic_ncols=True):
                        sample_meta = ds.samples[idx]
                        key = sample_meta["key"]
                        try:
                            _, landmarks, label, meta = ds[idx]
                        except Exception as exc:
                            logger.error(f"Lỗi trích xuất video '{sample_meta['path']}': {exc}")
                            stats["failed"].append({"key": key, "error": str(exc)})
                            continue

                        done[key] = self._pack_sample_entry(key, split, sample_meta, landmarks, label, meta)
                        stats["new"] += 1
                        stats["frames"] += int(landmarks.shape[0])
                        since_save += 1

                        if since_save >= self.cfg.cache_save_every:
                            cache["num_samples"] = len(done)
                            self.atomic_save(cache, path, sync_disk=False)
                            since_save = 0
            except KeyboardInterrupt:
                stats["interrupted"] = True
                logger.warning("Quá trình trích xuất bị ngắt bởi người dùng. Đang lưu các mẫu đã hoàn tất...")
            finally:
                cache["num_samples"] = len(done)
                self.atomic_save(cache, path, sync_disk=True)
                stats["seconds"] = time.time() - t0
        else:
            logger.info(f"Split '{split}' đã đủ 100% mẫu trong {path.name}. Bỏ qua trích xuất.")

        stats["size_mb"] = path.stat().st_size / (1024 * 1024) if path.is_file() else 0.0
        return stats

    def fit(self) -> Dict[str, Any]:
        """Trích xuất toàn bộ các split được chỉ định trong cfg.cache_splits."""
        summary: Dict[str, Any] = {}
        start = time.time()
        for split in self.cfg.cache_splits:
            stats = self.extract_split(split)
            stats["verify"] = self.verify_cache_file(self.cache_dir / f"{split}.pt")
            summary[split] = stats
            if stats["interrupted"]:
                break

        logger.info("=" * 70)
        logger.info(f"HOÀN TẤT TRÍCH XUẤT CACHE sau {(time.time() - start) / 60:.2f} phút.")
        for split, st in summary.items():
            vf = st["verify"]
            speed = (st["new"] / st["seconds"]) if st["seconds"] > 0 else 0.0
            fps_ = (st["frames"] / st["seconds"]) if st["seconds"] > 0 else 0.0
            full_match = vf['ok'] and (vf['num_samples'] == st['total'])
            logger.info(
                f"  [{split:5s}] tổng={st['total']} | mới={st['new']} | lỗi={len(st['failed'])} | "
                f"{speed:.2f} video/s ({fps_:.0f} frame/s) | {st.get('size_mb', 0.0):.1f} MB | "
                f"verify={'OK' if full_match else 'LỖI'} ({vf['num_samples']}/{st['total']} mẫu, dt={vf.get('time_interval')}s)"
            )
            for err in vf["errors"][:5]:
                logger.warning(f"      - {err}")
        return summary


In [6]:
# Cell 7: Bắt đầu chạy trích xuất toàn bộ Dataset sang các tệp .pt
config = ExtractConfig()

builder = LiteFatPtCacheBuilder(config)
cache_summary = builder.fit()


[2026-11-16 13:34:05][INFO] Đang khởi tạo PIPNet (resnet18 - 68 điểm 300W) trên thiết bị: CUDA


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[2026-11-16 13:34:06][INFO] Đã nạp xong trọng số mô hình PIPNet 68!
[2026-11-16 13:34:06][INFO] ======================================================================
[2026-11-16 13:34:06][INFO] KHỞI TẠO PIPELINE TRÍCH XUẤT .PT CHO LITEFAT DATASET (TURBO GPU + ASYNC I/O)
[2026-11-16 13:34:06][INFO] ======================================================================
[2026-11-16 13:34:06][INFO] Dataset root              : /root/.cache/huggingface/hub/datasets--nyvantran--data_processed_v1/snapshots/7b90773741e01f4d19ac6618d724690b5abff1ba
[2026-11-16 13:34:06][INFO] Thư mục cache (.pt)       : /content/LiteFat_v1/cache_pt
[2026-11-16 13:34:06][INFO] Các split                 : ['train', 'val', 'test']
[2026-11-16 13:34:06][INFO] Time interval             : 0.1s
[2026-11-16 13:34:06][INFO] Window sec                : None (None = toàn bộ video)
[2026-11-16 13:34:06][INFO] Letterbox size (w, h)     : (640, 640)
[2026-11-16 13:34:06][INFO] Face size (crop)          : (224, 224)
[2026-11-

Extracting [train]: 100%|██████████| 518/518 [1:25:34<00:00,  9.91s/video]


[2026-11-16 14:59:50][INFO] ============================================================
[2026-11-16 14:59:50][INFO] BẮT ĐẦU TRÍCH XUẤT SPLIT [VAL]: tổng 66 | đã có 0 | cần trích xuất 66
[2026-11-16 14:59:50][INFO] ============================================================


Extracting [val]: 100%|██████████| 66/66 [11:46<00:00, 10.70s/video]


[2026-11-16 15:11:37][INFO] ============================================================
[2026-11-16 15:11:37][INFO] BẮT ĐẦU TRÍCH XUẤT SPLIT [TEST]: tổng 65 | đã có 0 | cần trích xuất 65
[2026-11-16 15:11:37][INFO] ============================================================


Extracting [test]: 100%|██████████| 65/65 [12:44<00:00, 11.75s/video]


[2026-11-16 15:24:22][INFO] ======================================================================
[2026-11-16 15:24:22][INFO] HOÀN TẤT TRÍCH XUẤT CACHE sau 110.26 phút.
[2026-11-16 15:24:22][INFO]   [train] tổng=518 | mới=518 | lỗi=0 | 0.10 video/s (274 frame/s) | 1158.3 MB | verify=OK (518/518 mẫu, dt=0.1s)
[2026-11-16 15:24:22][INFO]   [val  ] tổng=66 | mới=66 | lỗi=0 | 0.09 video/s (255 frame/s) | 148.1 MB | verify=OK (66/66 mẫu, dt=0.1s)
[2026-11-16 15:24:22][INFO]   [test ] tổng=65 | mới=65 | lỗi=0 | 0.09 video/s (232 frame/s) | 145.9 MB | verify=OK (65/65 mẫu, dt=0.1s)


In [7]:
# Cell 8: Kiểm tra chi tiết mẫu dữ liệu & xác minh tương thích với train_1.ipynb
cache_root = Path(config.cache_dir)
sample_cache_file = next(
    (cache_root / f"{sp}.pt" for sp in ("test", "val", "train") if (cache_root / f"{sp}.pt").is_file()),
    None,
)

if sample_cache_file is not None:
    data = torch.load(sample_cache_file, map_location="cpu", weights_only=False)
    top_dt = data.get("time_interval")
    meta_dt = data.get("meta", {}).get("time_interval")
    print(
        f"File: {sample_cache_file.name} | Định dạng: v{data.get('format_version')} | "
        f"Số mẫu: {data.get('num_samples')} | time_interval: {top_dt}s (meta={meta_dt}s)"
    )

    first_key = next(iter(data["samples"]))
    sample = data["samples"][first_key]
    meta = sample["meta"]
    lbl = int(sample["label"])
    kss_state = "Drowsy" if lbl >= 7 else ("Mid-Fatigue" if lbl >= 5 else "Alert")

    print("=" * 64)
    print(f"Khóa mẫu (key)      : {first_key}")
    print(f"Video ID / Dataset  : {meta['video_id']} ({meta.get('dataset', 'unknown')})")
    print(f"Subject ID          : {meta.get('subject_id', 'unknown')}")
    print(f"Nhãn KSS (1-9)      : {lbl} ({kss_state}) | sample_weight={meta.get('sample_weight', 1.0):.4f}")
    print(f"FPS gốc             : {meta['fps']:.2f}")
    print(f"Thời lượng video    : {meta['duration_sec']:.2f}s (Tổng {meta['total_frames']} frames)")
    print(f"Số frame trích xuất : {meta['num_frames']} (~{meta['num_frames'] / max(1e-6, meta['duration_sec']):.2f} FPS)")
    print(f"Tỷ lệ phát hiện mặt : {meta['found_faces_ratio'] * 100:.1f}%")
    print(f"Shape landmarks     : {sample['landmarks'].shape} (Kỳ vọng: [T, 68, 3])")
    print(f"Shape crop_boxes    : {sample['crop_boxes'].shape} (Kỳ vọng: [T, 4])")
    print(f"Shape bboxes        : {sample['bboxes'].shape} (Kỳ vọng: [T, 5])")
    print(f"Shape found flags   : {sample['found'].shape} (Kỳ vọng: [T])")
    print(f"Shape target_indices: {meta['target_indices'].shape} (Kỳ vọng: [T])")
    print("=" * 64)
    print("Tọa độ crop_boxes 3 frame đầu tiên:")
    print(sample["crop_boxes"][:3].numpy())
else:
    print(f"Chưa tìm thấy file .pt nào trong {cache_root}.")


File: test.pt | Định dạng: v1 | Số mẫu: 65 | time_interval: 0.1s (meta=0.1s)
Khóa mẫu (key)      : test/uldd_A_R3D_A_S1_KSS4.mp4
Video ID / Dataset  : uldd_A_R3D_A_S1_KSS4 (UL-DD)
Subject ID          : uldd_A
Nhãn KSS (1-9)      : 4 (Alert) | sample_weight=1.0000
FPS gốc             : 60.00
Thời lượng video    : 240.00s (Tổng 14400 frames)
Số frame trích xuất : 2400 (~10.00 FPS)
Tỷ lệ phát hiện mặt : 100.0%
Shape landmarks     : torch.Size([2400, 68, 3]) (Kỳ vọng: [T, 68, 3])
Shape crop_boxes    : torch.Size([2400, 4]) (Kỳ vọng: [T, 4])
Shape bboxes        : torch.Size([2400, 5]) (Kỳ vọng: [T, 5])
Shape found flags   : torch.Size([2400]) (Kỳ vọng: [T])
Shape target_indices: torch.Size([2400]) (Kỳ vọng: [T])
Tọa độ crop_boxes 3 frame đầu tiên:
[[ 97  80 465 448]
 [ 98  81 464 448]
 [ 98  82 463 447]]


In [9]:
# Cell 9: Nén Cache .pt & Tùy chọn Sao lưu lên Google Drive
import shutil
from pathlib import Path

_cache_dir = Path(config.cache_dir if "config" in globals() else "/content/LiteFat_v1/cache_pt")
_zip_name = "litefat_v1_cache_pt"

if _cache_dir.exists():
    archive_path = shutil.make_archive(
        _zip_name,
        "zip",
        root_dir=_cache_dir.parent,
        base_dir=_cache_dir.name,
    )
    print(f"Đã nén toàn bộ cache .pt tại: {archive_path}")

    BACKUP_TO_DRIVE = True  # Đổi thành False nếu không cần sao lưu lên Google Drive
    if BACKUP_TO_DRIVE:
        try:
            from google.colab import drive

            drive_root = Path("/content/drive/MyDrive")
            if not drive_root.exists():
                drive.mount("/content/drive")
            dest_dir = drive_root / "LiteFat_v1"
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest_zip = dest_dir / Path(archive_path).name
            shutil.copy2(archive_path, dest_zip)
            print(f"Đã sao lưu cache lên Google Drive: {dest_zip}")
        except ImportError:
            print("Không phải môi trường Google Colab, bỏ qua bước sao lưu Google Drive.")
else:
    print(f"Không tìm thấy thư mục cache: {_cache_dir}")


Mounted at /content/drive
Đã sao lưu cache lên Google Drive: /content/drive/MyDrive/LiteFat_v1/litefat_v1_cache_pt.zip
